In [79]:
import os
import json
import polars as pl

In [80]:
script_path = os.getcwd()
project_path = os.path.join(script_path, '..')
features_dir = os.path.join(project_path, 'data', 'features')
models_pricing_path = os.path.join(features_dir, 'models_pricing.csv')

In [81]:
models_pricing_df = pl.read_csv(models_pricing_path)

In [82]:
models_pricing_df

model_name,input_cost,output_cost
str,f64,f64
"""gpt-4o""",2.5,10.0
"""gpt-4o-mini""",0.15,0.6
"""text-embedding-3-small""",0.02,null


In [83]:
feature_metadata_df_dict = {}

for filename in os.listdir(features_dir):
    name, extension = os.path.splitext(filename)

    if extension == ".json":
        file_path = os.path.join(features_dir, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            feature_metadata_dict = json.load(f)

        feature_metadata_df_dict[name] = pl.DataFrame(list(feature_metadata_dict.values()))


In [88]:
print(f'Estimated cost feature generation per processing unit (observation/comment):')

relative_cost_list = []

for feature_name in feature_metadata_df_dict.keys():  

    feature_metadata_df = feature_metadata_df_dict[feature_name].join(models_pricing_df, on='model_name', how='left') 
    
    if 'embeddings' not in feature_name:
        feature_metadata_df = feature_metadata_df.with_columns(
            (pl.col('input_tokens')*pl.col('input_cost')/1e6 + pl.col('output_tokens')*pl.col('output_cost')/1e6).alias('cost')
        )
    else:
        feature_metadata_df = feature_metadata_df.with_columns(
            (pl.col('input_tokens')*pl.col('input_cost')/1e6).alias('cost')
        )

    cost = feature_metadata_df['cost'].sum()
    sample_size = feature_metadata_df.shape[0]
    
    relative_cost = cost/sample_size
    relative_cost_list.append(relative_cost)

    print(f'- Cost of {feature_name}: {round(relative_cost, 6)} $ / processed unit')

total_relative_cost = sum(relative_cost_list)

# TODO: add cost of embeddings generation

Estimated cost feature generation per processing unit (observation/comment):
- Cost of argument_quality_score_metadata: 0.000113 $ / processed unit
- Cost of content_relevance_score_metadata: 8.9e-05 $ / processed unit
- Cost of discourse_tone_metadata: 0.000109 $ / processed unit
- Cost of dominant_frame_metadata: 0.000118 $ / processed unit
- Cost of embeddings_metadata: 2e-06 $ / processed unit
- Cost of political_stance_metadata: 0.00012 $ / processed unit
- Cost of sentiment_score_metadata: 0.000118 $ / processed unit


In [89]:
total_relative_cost

0.000668362909090909